# Inférence & évaluation -- classificateur de cellules sanguines (12 classes)

Ce notebook est **séparé du notebook d'entraînement** (`cell_classifier_12classes.ipynb`). Il suppose que
le modèle a déjà été entraîné et que ses checkpoints (`.keras`) sont accessibles (via `/kaggle/working`
dans la même session, ou via un dataset Kaggle attaché en `/kaggle/input/...`).

**2 usages :**
1. **Classer une image de cellule** que vous fournissez (chemin d'un fichier uploadé dans un dataset Kaggle,
   ou widget d'upload interactif) → renvoie la classe prédite + le statut à 3 états (`Test bon` /
   `Résultat indéterminé` / `Test pas bon`)
2. **Évaluer les performances du modèle** sur le jeu de test (accuracy, RMSE/score de Brier, matrice de
   confusion, rapport détaillé par classe)

## 1. Configuration

In [ ]:
from pathlib import Path
import os
import json
import types
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import cv2

# Chemins des datasets (identiques au notebook d'entraînement -- nécessaires pour
# reconstruire le split train/val/test de façon reproductible et évaluer sur le TEST)
LMU_DIR = Path("/kaggle/input/datasets/jfonlupt/pkg-aml-cytomorphology-lmu/PKG-AML-Cytomorphology_LMU")
PBC_DIR = Path("/kaggle/input/datasets/jfonlupt/pbc-dataset-normal-dib/PBC_dataset_normal_DIB/PBC_dataset_normal_DIB")
ANNOTATIONS_FILE = LMU_DIR / "annotations.dat"

IMG_SIZE = 400
RANDOM_SEED = 42          # identique à l'entraînement : garantit le même split train/val/test
BATCH_SIZE = 16

# Emplacement exact du modèle enregistré comme Kaggle Model (jfonlupt/026-08-14-cell-classifieur)
MODEL_DIR = Path("/kaggle/input/models/jfonlupt/026-08-14-cell-classifieur/tensorflow1/default/1")

# Emplacements de repli si MODEL_DIR ne contient pas (ou plus) les fichiers attendus
# (ex: nouvelle version du modèle re-publiée sous un autre numéro de version, ou
# session en cours dont les checkpoints n'ont pas encore été publiés comme Model)
FALLBACK_SEARCH_DIRS = ["/kaggle/working", "/kaggle/input"]

def find_artifact(filename):
    """Cherche un fichier d'abord dans MODEL_DIR (emplacement exact connu), puis en
    repli dans le disque de travail courant et les autres inputs Kaggle attachés."""
    direct = MODEL_DIR / filename
    if direct.exists():
        return direct
    for d in FALLBACK_SEARCH_DIRS:
        matches = sorted(Path(d).rglob(filename)) if Path(d).exists() else []
        if matches:
            return matches[0]
    return None


MODEL_PHASE2 = find_artifact("best_model_phase2.keras")
MODEL_PHASE1 = find_artifact("best_model_phase1.keras")
THRESHOLDS_FILE = find_artifact("thresholds.json")  # probablement absent -- recalibré section 6 si besoin

print("best_model_phase2.keras trouvé :", MODEL_PHASE2)
print("best_model_phase1.keras trouvé :", MODEL_PHASE1)
print("thresholds.json trouvé         :", THRESHOLDS_FILE)


## 2. Classes cibles et correspondance (identique à l'entraînement)

In [ ]:
TARGET_CLASSES = [
    "Neutrophil (segmented)", "Lymphocyte", "Eosinophil", "Myeloblast",
    "Monocyte", "Platelet", "Neutrophil (band)", "Erythroblast",
    "Basophil", "Myelocyte", "Metamyelocyte", "Promyelocyte",
]
CLASS_TO_IDX = {c: i for i, c in enumerate(TARGET_CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
N_CLASSES = len(TARGET_CLASSES)

LMU_TO_TARGET = {
    "NGS": "Neutrophil (segmented)", "NGB": "Neutrophil (band)",
    "LYT": "Lymphocyte", "LYA": "Lymphocyte", "EOS": "Eosinophil",
    "MYO": "Myeloblast", "MON": "Monocyte", "EBO": "Erythroblast",
    "BAS": "Basophil", "MYB": "Myelocyte", "MMZ": "Metamyelocyte",
    "PMO": "Promyelocyte", "PMB": "Promyelocyte",
}
PBC_TO_TARGET = {
    "BA": "Basophil", "EO": "Eosinophil", "ERB": "Erythroblast",
    "LY": "Lymphocyte", "MO": "Monocyte", "BNE": "Neutrophil (band)",
    "SNE": "Neutrophil (segmented)", "NEUTROPHIL": "Neutrophil (segmented)",
    "PMY": "Promyelocyte", "MY": "Myelocyte", "MMY": "Metamyelocyte",
    "PLATELET": "Platelet",
}


## 3. Chargement du modèle entraîné

On charge en priorité le modèle de la **Phase 2** (fine-tuné), sinon on retombe sur la **Phase 1**
(tête seule). `rmse_metric` doit être redéfinie ici car c'est une métrique personnalisée non
sérialisable nativement par Keras.

In [ ]:
def rmse_metric(y_true, y_pred):
    """RMSE multiclasse (racine du score de Brier) -- identique à l'entraînement."""
    return tf.sqrt(tf.reduce_mean(tf.square(y_true - y_pred)))


MODEL_PATH = MODEL_PHASE2 or MODEL_PHASE1
if MODEL_PATH is None:
    raise FileNotFoundError(
        "Aucun modèle entraîné trouvé (ni best_model_phase2.keras, ni best_model_phase1.keras). "
        "Vérifiez que le notebook d'entraînement a bien tourné et que ses fichiers de sortie "
        "sont accessibles (même session Kaggle, ou dataset attaché en /kaggle/input/...)."
    )

model = tf.keras.models.load_model(str(MODEL_PATH), custom_objects={"rmse_metric": rmse_metric})
print(f"Modèle chargé depuis : {MODEL_PATH}")
model.summary()


## 4. Reconstruction du split train/val/test

Nécessaire pour (a) évaluer sur le même jeu de **test** qu'à l'entraînement (jamais vu par le modèle),
et (b) recalibrer les seuils de confiance si `thresholds.json` n'a pas été sauvegardé. Comme le split
utilise le même `RANDOM_SEED` et les mêmes données sources, il est **strictement identique** à celui
utilisé pendant l'entraînement (à condition que les datasets sources n'aient pas changé entre-temps).

In [ ]:
from sklearn.model_selection import train_test_split


def extract_pbc_subclass(filename_stem):
    return filename_stem.split("_")[0].upper()


def load_LMU_labeled():
    df = pd.read_csv(
        ANNOTATIONS_FILE, sep=" ", header=None,
        names=["relative_path", "class_code", "re_annotation", "re_re_annotation"]
    )
    df["path"] = df["relative_path"].apply(lambda p: LMU_DIR / p)
    df["exists"] = df["path"].apply(lambda p: p.exists())

    def final_code(row):
        for v in (row["re_re_annotation"], row["re_annotation"]):
            if pd.notna(v) and str(v) != "nan":
                return v
        return row["class_code"]

    df["source_code"] = df.apply(final_code, axis=1)
    df["target_label"] = df["source_code"].map(LMU_TO_TARGET)
    return df[df["exists"] & df["target_label"].notna()][["path", "target_label"]]


def load_PBC_labeled():
    rows = []
    for class_dir in sorted(PBC_DIR.iterdir()):
        if class_dir.is_dir():
            for f in class_dir.iterdir():
                if f.suffix.lower() in (".png", ".jpg", ".jpeg", ".tif", ".tiff"):
                    rows.append({"path": f, "source_code": extract_pbc_subclass(f.stem)})
    df = pd.DataFrame(rows)
    df["target_label"] = df["source_code"].map(PBC_TO_TARGET)
    return df[df["target_label"].notna()][["path", "target_label"]]


data = pd.concat([load_LMU_labeled(), load_PBC_labeled()], ignore_index=True)
data["label_idx"] = data["target_label"].map(CLASS_TO_IDX)

train_df, temp_df = train_test_split(data, test_size=0.30, stratify=data["target_label"], random_state=RANDOM_SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["target_label"], random_state=RANDOM_SEED)

print(f"Train : {len(train_df)} | Validation : {len(val_df)} | Test : {len(test_df)}")


## 5. Prétraitement image et fonction d'inférence sur une seule photo

**Comment fournir votre image sur Kaggle :**
- **Option A (recommandée pour Kaggle)** : ajoutez votre image via *Add Input > Upload* (crée un mini
  dataset), puis renseignez son chemin dans `IMAGE_PATH` ci-dessous (visible dans le panneau *Input* à
  droite, sous `/kaggle/input/<nom-du-dataset>/...`).
- **Option B (session interactive uniquement, ne fonctionne pas en mode Commit)** : widget d'upload
  direct dans le notebook, fourni juste après.

In [ ]:
def _read_image_cv2(path_bytes):
    path_str = path_bytes.numpy().decode("utf-8")
    img = cv2.imread(path_str, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"Impossible de lire l'image : {path_str}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return img.astype(np.float32) / 255.0


def load_and_preprocess(path, label):
    img = tf.py_function(func=_read_image_cv2, inp=[path], Tout=tf.float32)
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return img, label


def classify_status(confidence, t_low, t_high):
    if confidence >= t_high:
        return "Test bon"
    elif confidence < t_low:
        return "Test pas bon"
    return "Résultat indéterminé"


def predict_cell(image_path, model=model, t_low=None, t_high=None):
    """Prédit le type d'une cellule à partir d'une image, avec sortie à 3 états."""
    img, _ = load_and_preprocess(str(image_path), tf.zeros(N_CLASSES))
    proba = model.predict(tf.expand_dims(img, 0), verbose=0)[0]
    pred_idx = int(np.argmax(proba))
    confidence = float(np.max(proba))
    status = classify_status(confidence, t_low, t_high) if (t_low is not None and t_high is not None) else "Seuils non calibrés"
    return {
        "classe_predite": IDX_TO_CLASS[pred_idx],
        "confiance": round(confidence, 4),
        "statut": status,
        "top3": sorted(
            [(IDX_TO_CLASS[i], round(float(p), 4)) for i, p in enumerate(proba)],
            key=lambda x: -x[1]
        )[:3],
        "image": img.numpy(),
    }


## 6. Seuils de confiance (chargés si sauvegardés, sinon recalibrés)

In [ ]:
TARGET_PRECISION = 0.999
REJECT_FLOOR = 0.5
MIN_SUPPORT_CALIBRATION = 30


def calibrate_thresholds(y_true, y_pred, confidences, target_precision=TARGET_PRECISION,
                          reject_floor=REJECT_FLOOR, min_support=MIN_SUPPORT_CALIBRATION):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred); confidences = np.asarray(confidences)
    order = np.argsort(-confidences)
    conf_sorted = confidences[order]
    correct_sorted = (y_true[order] == y_pred[order]).astype(int)
    cum_precision = np.cumsum(correct_sorted) / np.arange(1, len(correct_sorted) + 1)
    cum_count = np.arange(1, len(correct_sorted) + 1)

    def last_threshold_meeting(floor):
        for i in range(len(cum_precision) - 1, -1, -1):
            if cum_count[i] >= min_support and cum_precision[i] >= floor:
                return conf_sorted[i]
        return None

    t_high = last_threshold_meeting(target_precision)
    t_low = last_threshold_meeting(reject_floor)
    if t_high is None:
        t_high = 1.01
    if t_low is None:
        t_low = float(conf_sorted.min())
    return float(t_high), float(min(t_low, t_high))


if THRESHOLDS_FILE is not None:
    thresholds = json.load(open(THRESHOLDS_FILE))
    T_HIGH, T_LOW = thresholds["t_high"], thresholds["t_low"]
    print(f"Seuils rechargés depuis {THRESHOLDS_FILE} : t_high={T_HIGH:.4f} | t_low={T_LOW:.4f}")
else:
    print("Aucun thresholds.json trouvé -> recalibrage sur le jeu de validation...")

    def make_dataset(df):
        paths = df["path"].astype(str).values
        labels = tf.one_hot(df["label_idx"].values, N_CLASSES)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
        return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    val_ds = make_dataset(val_df)
    val_proba = model.predict(val_ds, verbose=0)
    val_pred_idx = val_proba.argmax(axis=1)
    val_confidence = val_proba.max(axis=1)
    T_HIGH, T_LOW = calibrate_thresholds(val_df["label_idx"].values, val_pred_idx, val_confidence)

    # Sauvegarde pour ne plus avoir à recalculer la prochaine fois
    with open("/kaggle/working/thresholds.json", "w") as f:
        json.dump({"t_high": T_HIGH, "t_low": T_LOW}, f)
    print(f"Seuils recalibrés et sauvegardés : t_high={T_HIGH:.4f} | t_low={T_LOW:.4f}")


## 7. Insertion d'une photo de cellule à déterminer

**Renseignez `IMAGE_PATH` ci-dessous** avec le chemin de votre image (voir Option A section 5).

In [ ]:
IMAGE_PATH = "/kaggle/input/votre-dataset-uploade/votre_image.jpg"  # <-- à modifier

if os.path.exists(IMAGE_PATH):
    result = predict_cell(IMAGE_PATH, t_low=T_LOW, t_high=T_HIGH)

    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(result["image"])
    ax.axis("off")
    ax.set_title(f"{result['classe_predite']}\n({result['statut']}, confiance={result['confiance']:.1%})")
    plt.show()

    print(f"Classe prédite : {result['classe_predite']}")
    print(f"Statut         : {result['statut']}")
    print(f"Confiance      : {result['confiance']:.2%}")
    print("Top 3 :")
    for cls, p in result["top3"]:
        print(f"  - {cls} : {p:.2%}")
else:
    print(f"IMAGE_PATH introuvable : {IMAGE_PATH}")
    print("Modifiez la variable IMAGE_PATH ci-dessus avec le chemin réel de votre image.")


### Option B -- upload interactif (session interactive uniquement)

Ne fonctionne pas en mode `Save & Run All (Commit)` : à utiliser seulement si vous exécutez ce
notebook cellule par cellule dans une session Kaggle ouverte.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    uploader = widgets.FileUpload(accept="image/*", multiple=False)
    display(uploader)
    print("Uploadez une image ci-dessus, puis exécutez la cellule suivante.")
except ImportError:
    print("ipywidgets non disponible dans cet environnement -- utilisez l'Option A (IMAGE_PATH).")


In [ ]:
# À exécuter après avoir uploadé une image via le widget ci-dessus
def _extract_uploaded_file(value):
    """Compatible ipywidgets < 8 (value = dict {nom: infos}) et >= 8
    (value = tuple/list de dicts) -- l'API a changé entre les versions."""
    if isinstance(value, dict):
        return list(value.values())[0]
    return value[0]


if "uploader" in dir() and len(uploader.value) > 0:
    uploaded_file = _extract_uploaded_file(uploader.value)
    tmp_path = "/kaggle/working/_uploaded_cell_image.png"
    with open(tmp_path, "wb") as f:
        f.write(bytes(uploaded_file["content"]))  # bytes() gère aussi bien 'bytes' que 'memoryview'

    result = predict_cell(tmp_path, t_low=T_LOW, t_high=T_HIGH)
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(result["image"])
    ax.axis("off")
    ax.set_title(f"{result['classe_predite']}\n({result['statut']}, confiance={result['confiance']:.1%})")
    plt.show()
    print(result)
else:
    print("Aucune image uploadée pour l'instant (ou Option A utilisée à la place).")


## 8. Évaluation des performances du modèle (jeu de test)

Le jeu de test reconstruit en section 4 n'a jamais été vu par le modèle pendant l'entraînement ni
pendant la calibration des seuils -- c'est la mesure la plus fiable de la performance réelle.

In [ ]:
def make_dataset(df):
    paths = df["path"].astype(str).values
    labels = tf.one_hot(df["label_idx"].values, N_CLASSES)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


def brier_rmse(y_true_idx, proba_matrix, n_classes):
    onehot = np.eye(n_classes)[y_true_idx]
    return float(np.sqrt(np.mean((onehot - proba_matrix) ** 2)))


test_ds = make_dataset(test_df)
test_proba = model.predict(test_ds, verbose=0)
test_pred_idx = test_proba.argmax(axis=1)
test_confidence = test_proba.max(axis=1)
test_true_idx = test_df["label_idx"].values

test_rmse = brier_rmse(test_true_idx, test_proba, N_CLASSES)
test_accuracy = (test_pred_idx == test_true_idx).mean()
print(f"RMSE (score de Brier) sur le test : {test_rmse:.4f}")
print(f"Accuracy globale sur le test      : {test_accuracy*100:.2f}%")

statuses = np.array([classify_status(c, T_LOW, T_HIGH) for c in test_confidence])
print()
print(f"{'Statut':22s} {'n':>6s} {'% du test':>10s} {'précision':>10s}")
for s in ["Test bon", "Résultat indéterminé", "Test pas bon"]:
    mask = statuses == s
    n_s = mask.sum()
    acc_s = (test_true_idx[mask] == test_pred_idx[mask]).mean() if n_s > 0 else float("nan")
    print(f"{s:22s} {n_s:6d} {n_s/len(test_df)*100:9.1f}% {acc_s*100:9.2f}%")


### Matrice de confusion

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, title, normalize=True):
    n = len(class_names)
    cm = np.zeros((n, n), dtype=float)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        cm_display = cm / row_sums
        fmt = "{:.0%}"
    else:
        cm_display = cm
        fmt = "{:.0f}"

    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(cm_display, cmap="Blues", vmin=0, vmax=max(cm_display.max(), 1e-6))
    ax.set_xticks(range(n)); ax.set_xticklabels(class_names, rotation=90)
    ax.set_yticks(range(n)); ax.set_yticklabels(class_names)
    ax.set_xlabel("Classe prédite"); ax.set_ylabel("Classe réelle")
    ax.set_title(title)
    for i in range(n):
        for j in range(n):
            if cm[i, j] > 0:
                ax.text(j, i, fmt.format(cm_display[i, j]), ha="center", va="center", fontsize=7,
                        color="white" if cm_display[i, j] > cm_display.max() / 2 else "black")
    plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()


plot_confusion_matrix(test_true_idx, test_pred_idx, TARGET_CLASSES, "Matrice de confusion -- ensemble du test")

bon_mask = statuses == "Test bon"
if bon_mask.sum() > 0:
    plot_confusion_matrix(test_true_idx[bon_mask], test_pred_idx[bon_mask], TARGET_CLASSES,
                           "Matrice de confusion -- bucket 'Test bon' uniquement")


### Rapport détaillé par classe (precision / recall / f1-score)

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(
    test_true_idx, test_pred_idx,
    labels=list(range(N_CLASSES)), target_names=TARGET_CLASSES,
    zero_division=0, output_dict=True,
)
report_df = pd.DataFrame(report).T.round(3)
report_df


## Conclusion

- Le modèle est rechargé automatiquement (Phase 2 si disponible, sinon Phase 1).
- Les seuils de confiance (`t_high`/`t_low`) sont rechargés s'ils ont été sauvegardés à l'entraînement,
  sinon recalibrés à la volée sur le jeu de validation reconstruit.
- La section 7 permet de classer n'importe quelle nouvelle image de cellule (400x400 recommandé,
  redimensionnement automatique sinon) avec la sortie à 3 états.
- La section 8 donne une évaluation complète et honnête sur le jeu de **test**, jamais vu par le modèle.